# 결과 요약 (2026-11-03)
따라하기 01~04와 확장 연구의 핵심 결과를 한곳에 모은다. 이 노트북은 계산을 새로 하지 않고, 각 노트북이 남긴 파일만 읽는다.
그래서 **먼저 따라하기 01~04와 `연구/확장연구_20261103.ipynb`를 실행해야** 이 노트북의 숫자가 최신이 된다(각 칸 위에 어느 노트북이 만든 파일인지 적었다).

| 파일 | 만든 노트북 |
|---|---|
| `결과_월수익률/01_우동호2023.pkl`, `01_원문대조.csv` | `따라하기/01_우동호2023.ipynb` |
| `결과_월수익률/02_NH밸류.pkl`, `02_원문대조_그림19.csv`, `02_원문주장.csv` | `따라하기/02_NH_밸류.ipynb` |
| `결과_월수익률/03_NH퀄리티로우볼.pkl`, `03_원문판정.csv` | `따라하기/03_NH_퀄리티로우볼.ipynb` |
| `결과_월수익률/04_신마법공식.pkl`, `04_NovyMarx대조.csv` | `따라하기/04_신마법공식.ipynb` |
| `연구/결과/확장연구_20261103.pkl`, `.csv` | `연구/확장연구_20261103.ipynb` |

데이터: DataGuide에서 새로 통째로 받은 일간 파일(`data\통합_전체_202611`, 1988-01-04~2026-09-30, 코스피+코스닥 상장폐지 포함). 데이터 규칙과 한계는 `README.md`.
줄임말: EW(동일가중), VW(시가총액가중), MDD(최대 낙폭), IR(정보비율), NW(뉴이-웨스트 t), LS(롱숏), TR(배당 포함 수정주가).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "공통"))
import pandas as pd
import 평가
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
폴더 = 평가.결과폴더
읽기csv = lambda n, **k: pd.read_csv(폴더 / n, encoding="utf-8-sig", **k)
for f in sorted(폴더.glob("*.*")) + sorted((Path.cwd() / "결과").glob("확장연구_20261103.*")):
    print(f"{pd.Timestamp(f.stat().st_mtime, unit='s', tz='UTC').tz_convert('Asia/Seoul'):%Y-%m-%d %H:%M}  {f.name}")

## 1. 우동호 외(2023) 재현 — 원문 표 2~7 vs 우리 (본문 설정)
2004-04~2022-03, 4월 첫 거래일 리밸런싱, 상위 30종목 동일가중, 벤치마크 코스피(원문 p.513 "벤치마크 수익률은 KOSPI수익률을 사용한다"). 누적·MDD·표준편차는 소수(9.8 = 980%).

In [ ]:
읽기csv("01_원문대조.csv", header=[0, 1], index_col=[0, 1])

### 1-1. 우리 결과: 본문·확인판(발표일 규칙)·A070480 빼기, EW·VW

In [ ]:
d = pd.read_pickle(폴더 / "01_우동호2023.pkl")
R = d["R"].copy(); R[("KOSPI", "-")] = d["F"]["시장"]
평가.성과표(R.dropna(how="all"), d["F"])[["개월", "누적", "연환산", "표준편차(연)", "샤프", "MDD", "정보비율", "CAPM α(월)", "Carhart α(월)", "Carhart t(NW)"]]

## 2. NH 밸류(2018) — 그림 19 위험조정수익률, 원문 주장

In [ ]:
display(읽기csv("02_원문대조_그림19.csv", index_col=0))
읽기csv("02_원문주장.csv", index_col=0)

## 3. NH 퀄리티·로우볼(2018) — 원문 주장 판정 (다시 돌린 결과)
'확인판(자본>0만)' 칸은 ROE·부채비율을 자본이 0보다 큰 종목만으로 계산한 것이다. 두 칸이 다르면 그 판정은 음수 자본 처리에 달려 있다.

In [ ]:
읽기csv("03_원문판정.csv", index_col=0)

## 4. 신마법공식(PBR + GP/A)과 Novy-Marx(2013) 표 7 참고판

In [ ]:
d = pd.read_pickle(폴더 / "04_신마법공식.pkl")
R = d["R"].copy(); R[("KOSPI", "-")] = d["F"]["시장"]
display(평가.성과표(R.dropna(how="all"), d["F"])[["개월", "누적", "연환산", "샤프", "MDD", "정보비율", "Carhart α(월)", "Carhart t(NW)"]])
읽기csv("04_NovyMarx대조.csv", header=[0, 1], index_col=0)

## 5. 따라하기 전체 요약표: 전략 × 가중 × 기간 (`평가.요약`)

In [ ]:
요약 = 평가.요약()
요약

## 6. 확장 연구 핵심 결과
우동호 유니버스, 배당 포함 가격, 1990-04~2026-09. 주 벤치마크 = 유니버스 시가총액 가중(비용 0). 자세한 표와 설정·근거는 `연구/확장연구_20261103.ipynb`.

In [ ]:
x = pd.read_pickle(Path.cwd() / "결과" / "확장연구_20261103.pkl")
표 = x["표"]
전체 = 표.xs("전체", level="기간")
기본 = 전체.xs("비용기본", level="비용")
print(f"전략 {len(x['전략들'])}개, 결과표 {len(표)}줄")
display(표.loc[[i for i in 표.index if i[0].startswith("벤치:")]].xs("전체", level="기간").droplevel([1, 2, 3]).style.set_caption("벤치마크(전체 기간)").format(precision=4))
기본.loc["EY+ROC 순위합산"]["샤프"].unstack("주기").style.set_caption("마법공식(EY+ROC 순위합산): 비중 × 주기 샤프, 전체 기간, 기본 비용").format(precision=3)

### 6-1. 동일가중·12개월·기본 비용: 샤프 상위 10 전략과 기준(전체 기간)

In [ ]:
k = 기본.xs(("동일", 12), level=["비중", "주기"]).sort_values("샤프", ascending=False)
k = k.assign(순위=range(1, len(k) + 1))
pd.concat([k.head(10), k.loc[["EY+ROC 순위합산"]]]).pipe(lambda t: t[~t.index.duplicated()])  # 같은 전략이 두 번 나오면 하나만(감사 L11)

### 6-2. 비중 방식 × 주기: 전 전략 샤프 중앙값 / 거래비용 민감도(연환산 중앙값)

In [ ]:
전략줄 = lambda t: t.drop(index=[i for i in t.index if i[0].startswith("벤치:")])
display(전략줄(표.xs(("공통", "비용기본"), level=["기간", "비용"]))["샤프"].groupby(["비중", "주기"]).median().unstack("주기").style.set_caption("샤프 중앙값('공통' 기간: 모든 비중 방식이 같은 달부터)").format(precision=3))
전략줄(전체)["연환산"].groupby(["주기", "비용"]).median().unstack("비용")[["비용0", "비용기본", "비용2배"]].style.set_caption("연환산 수익률 중앙값").format(precision=4)

### 6-3. 기간별 일관성: 동일가중·12개월·기본 비용에서 유니버스 벤치마크보다 샤프가 높은 전략 수

In [ ]:
b = 표.xs(("벤치:유니버스(시총가중)", "-", 0, "비용0"), level=[0, 1, 2, 3])["샤프"]
s = 전략줄(표).xs(("동일", 12, "비용기본"), level=["비중", "주기", "비용"])["샤프"].unstack("기간")
pd.DataFrame({"전략 수": s.notna().sum(), "벤치마크보다 샤프 높음": (s.gt(b)).sum(), "벤치마크 샤프": b}).T

## 7. 남은 한계와 확인할 것
README '한계·미해결'과 `_진행기록_20261103.md` 맨 끝 '끝' 절을 본다.